# Complete Guide to Out-of-Distribution (OOD) Detection

This notebook provides a comprehensive introduction to OOD detection using `incerto`. We'll cover:

1. **Understanding OOD Detection** - What is OOD detection and why it matters
2. **Setup and Data Preparation** - Load FashionMNIST (ID) and MNIST (OOD)
3. **Train a Classification Model** - Base model for OOD detection
4. **Post-hoc OOD Detection Methods** - MSP, Energy, MaxLogit, ODIN, Mahalanobis, KNN
5. **Training-time OOD Methods** - Mixup, CutMix, Outlier Exposure, Energy Regularization
6. **Evaluation and Visualization** - AUROC, FPR@95TPR, ROC curves, score histograms
7. **Production Deployment** - Thresholding, save/load, inference pipeline
8. **Best Practices** - Method selection guidelines

**Expected Runtime**: ~3 min (MPS/CUDA), ~8 min (CPU)

## Part 1: Understanding OOD Detection

### What is OOD Detection?

Out-of-Distribution (OOD) detection identifies when test inputs differ significantly from training data. This is crucial for:
- **Safety**: Preventing confident but wrong predictions on unfamiliar data
- **Reliability**: Flagging inputs that require human review
- **Monitoring**: Detecting distribution shifts in production

### Methods in This Notebook

**Post-hoc Methods** (applied after training):

| Method | Key Idea | Cost |
|--------|----------|------|
| **MSP** | Maximum softmax probability | Very Low |
| **Energy** | Energy score from logits | Low |
| **MaxLogit** | Maximum logit value | Very Low |
| **ODIN** | Temperature + input perturbation | Medium |
| **Mahalanobis** | Distance in feature space | High (requires fit) |
| **KNN** | K-nearest neighbors in features | High (requires fit) |

**Training-time Methods** (integrated during training):

| Method | Key Idea |
|--------|----------|
| **Mixup** | Linear interpolation of inputs |
| **CutMix** | Cut and paste patches |
| **Outlier Exposure** | Train on auxiliary OOD data |
| **Energy Regularization** | Margin loss on energy scores |

### Experimental Setup

- **In-Distribution (ID)**: FashionMNIST (clothing items, 10 classes)
- **Out-of-Distribution (OOD)**: MNIST digits (visually similar but semantically different)
- **Goal**: Detect MNIST digits as OOD while accepting FashionMNIST as ID

## Part 2: Setup

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Post-hoc OOD detection methods
from incerto.ood import (
    MSP,
    Energy,
    MaxLogit,
    ODIN,
    Mahalanobis,
    KNN,
)

# Training-time OOD methods
from incerto.ood import (
    mixup_data,
    mixup_criterion,
    CutMix,
    OutlierExposureLoss,
)

# OOD metrics and visualization
from incerto.ood import (
    auroc,
    fpr_at_tpr,
    detection_accuracy,
    plot_roc,
    score_hist,
    compute_threshold_at_tpr,
    get_ood_predictions,
)

from incerto.utils import ConvNet, seed_everything

seed_everything(42)

# Device selection: CUDA > MPS (Apple Silicon) > CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

## Part 3: Data Preparation

We use **FashionMNIST** as in-distribution (ID) and **MNIST** as out-of-distribution (OOD).
Both have the same image format (28x28 grayscale) but different content.

In [ ]:
# Data transforms (FashionMNIST normalization)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))
])

# Load FashionMNIST (In-Distribution)
train_dataset = datasets.FashionMNIST('./data', train=True, download=True, transform=transform)
test_id_dataset = datasets.FashionMNIST('./data', train=False, download=True, transform=transform)

# Load MNIST (Out-of-Distribution)
test_ood_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)

# Optimized data loaders
num_workers = min(4, os.cpu_count() or 0)
pin_memory = device.type == "cuda"
loader_kwargs = dict(num_workers=num_workers, pin_memory=pin_memory, persistent_workers=num_workers > 0)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, **loader_kwargs)
test_id_loader = DataLoader(test_id_dataset, batch_size=512, shuffle=False, **loader_kwargs)
test_ood_loader = DataLoader(test_ood_dataset, batch_size=512, shuffle=False, **loader_kwargs)

print(f"Training (FashionMNIST): {len(train_dataset)}")
print(f"Test ID (FashionMNIST): {len(test_id_dataset)}")
print(f"Test OOD (MNIST): {len(test_ood_dataset)}")
print(f"DataLoader: num_workers={num_workers}, pin_memory={pin_memory}")

In [ ]:
# Visualize ID vs OOD samples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))

# Show FashionMNIST (ID)
fashion_labels = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
                  'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Boot']
for i in range(5):
    img, label = test_id_dataset[i]
    axes[0, i].imshow(img.squeeze(), cmap='gray')
    axes[0, i].set_title(f'ID: {fashion_labels[label]}')
    axes[0, i].axis('off')

# Show MNIST (OOD)
for i in range(5):
    img, label = test_ood_dataset[i]
    axes[1, i].imshow(img.squeeze(), cmap='gray')
    axes[1, i].set_title(f'OOD: digit {label}')
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('In-Distribution\n(FashionMNIST)', fontsize=11)
axes[1, 0].set_ylabel('Out-of-Distribution\n(MNIST)', fontsize=11)

plt.tight_layout()
plt.show()

## Part 4: Train Classification Model

We train a CNN on FashionMNIST. OOD detection methods will use this model's outputs.

In [ ]:
# Use incerto's ConvNet - fc1 is the penultimate layer (required for Mahalanobis/KNN)
model = ConvNet(num_classes=10, dropout_rate=0.0).to(device)
print(model)

In [ ]:
# Training function
def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for data, target in loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.cross_entropy(output, target)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
        total += target.size(0)
    
    return total_loss / len(loader), 100. * correct / total

# Train model
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
epochs = 10

print("Training model on FashionMNIST...")
for epoch in range(epochs):
    loss, acc = train_epoch(model, train_loader, optimizer, device)
    if (epoch + 1) % 2 == 0:
        print(f"  Epoch {epoch+1}/{epochs} - Loss: {loss:.4f}, Accuracy: {acc:.2f}%")

print("Done!")

## Part 5: Post-hoc OOD Detection Methods

Post-hoc methods are applied **after training** - they don't modify the model.

In [ ]:
# Initialize all OOD detectors
model.eval()

detectors = {
    'MSP': MSP(model),
    'Energy': Energy(model, temperature=1.0),
    'MaxLogit': MaxLogit(model),
    'ODIN': ODIN(model, temperature=1000.0, epsilon=0.002),
    'Mahalanobis': Mahalanobis(model, layer_name='fc1'),
    'KNN': KNN(model, k=10, layer_name='fc1'),
}

print("OOD Detectors initialized:")
for name, detector in detectors.items():
    print(f"  - {detector}")

In [ ]:
# Fit detectors that require training data (Mahalanobis, KNN)
print("Fitting detectors that require training data...")

# Use subset of training data for efficiency
train_subset = Subset(train_dataset, range(0, len(train_dataset), 10))  # Every 10th sample
train_subset_loader = DataLoader(train_subset, batch_size=256, shuffle=False, **loader_kwargs)

detectors['Mahalanobis'].fit(train_subset_loader)
print(f"  {detectors['Mahalanobis']}")

detectors['KNN'].fit(train_subset_loader)
print(f"  {detectors['KNN']}")

print("All detectors ready!")

In [ ]:
# Helper function to score DataLoaders
def score_loader(detector, loader, device, needs_grad=False):
    """
    Compute OOD scores for all samples in a DataLoader.
    
    Args:
        detector: OOD detector instance
        loader: DataLoader with samples
        device: torch device
        needs_grad: If True, enable gradients (required for ODIN)
    
    Returns:
        Tensor of scores
    """
    detector.model.eval()
    all_scores = []
    
    for data, _ in loader:
        data = data.to(device)
        if needs_grad:
            # ODIN needs gradients for input perturbation
            scores = detector.score(data)
        else:
            with torch.no_grad():
                scores = detector.score(data)
        all_scores.append(scores.cpu())
    
    return torch.cat(all_scores)

In [ ]:
# Compute OOD scores for all methods
print("Computing OOD scores...\n")

results = {}

for name, detector in detectors.items():
    print(f"Running {name}...")
    
    # ODIN needs gradients for input perturbation
    needs_grad = (name == 'ODIN')
    
    # Compute scores (higher = more OOD-like)
    id_scores = score_loader(detector, test_id_loader, device, needs_grad=needs_grad)
    ood_scores = score_loader(detector, test_ood_loader, device, needs_grad=needs_grad)
    
    results[name] = {
        'id_scores': id_scores,
        'ood_scores': ood_scores,
    }
    
    print(f"  ID mean: {id_scores.mean():.4f}, OOD mean: {ood_scores.mean():.4f}")

print("\nAll scores computed!")

## Part 6: Evaluation and Visualization

We evaluate using `incerto`'s built-in OOD metrics:
- **AUROC**: Area under ROC curve (higher is better, 1.0 is perfect)
- **FPR@95TPR**: False positive rate when 95% of OOD samples are detected
- **Detection Accuracy**: Accuracy at a threshold accepting 95% of ID samples

In [ ]:
# Compute metrics for each method using incerto's built-in functions
print("OOD Detection Results:")
print("=" * 65)
print(f"{'Method':<15} {'AUROC':>10} {'FPR@95TPR':>12} {'Det. Acc':>12}")
print("-" * 65)

metrics_results = {}

for name, scores in results.items():
    id_s = scores['id_scores']
    ood_s = scores['ood_scores']
    
    # Use incerto's built-in metrics
    auc = auroc(id_s, ood_s)
    fpr = fpr_at_tpr(id_s, ood_s, tpr=0.95)
    det_acc = detection_accuracy(id_s, ood_s, id_accept_rate=0.95)
    
    metrics_results[name] = {'auroc': auc, 'fpr': fpr, 'det_acc': det_acc}
    print(f"{name:<15} {auc:>10.4f} {fpr:>12.4f} {det_acc:>12.4f}")

print("=" * 65)
best = max(metrics_results, key=lambda k: metrics_results[k]['auroc'])
print(f"\nBest method: {best} (AUROC = {metrics_results[best]['auroc']:.4f})")

In [ ]:
# Visualize score distributions using incerto's score_hist
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for idx, (name, scores) in enumerate(results.items()):
    ax = axes[idx]
    score_hist(scores['id_scores'], scores['ood_scores'], ax=ax, bins=50)
    auc = metrics_results[name]['auroc']
    ax.set_title(f'{name} (AUROC: {auc:.3f})')

plt.suptitle('OOD Score Distributions', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Plot ROC curves using incerto's plot_roc
fig, ax = plt.subplots(figsize=(10, 8))

for name, scores in results.items():
    auc = metrics_results[name]['auroc']
    plot_roc(scores['id_scores'], scores['ood_scores'], 
             label=f'{name} (AUROC: {auc:.3f})', ax=ax)

ax.set_title('ROC Curves for OOD Detection Methods', fontsize=14)
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

names = list(metrics_results.keys())
aurocs = [metrics_results[n]['auroc'] for n in names]
fprs = [metrics_results[n]['fpr'] for n in names]
det_accs = [metrics_results[n]['det_acc'] for n in names]

# AUROC (higher is better)
ax = axes[0]
bars = ax.bar(names, aurocs, color='steelblue', alpha=0.8)
ax.set_ylabel('AUROC (higher is better)')
ax.set_title('AUROC')
ax.set_ylim(0.5, 1.0)
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Random')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
for bar, val in zip(bars, aurocs):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9)

# FPR@95TPR (lower is better)
ax = axes[1]
bars = ax.bar(names, fprs, color='coral', alpha=0.8)
ax.set_ylabel('FPR@95TPR (lower is better)')
ax.set_title('FPR at 95% TPR')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
for bar, val in zip(bars, fprs):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9)

# Detection Accuracy (higher is better)
ax = axes[2]
bars = ax.bar(names, det_accs, color='seagreen', alpha=0.8)
ax.set_ylabel('Detection Accuracy')
ax.set_title('Detection Accuracy @95% ID Acceptance')
ax.set_ylim(0.5, 1.0)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
for bar, val in zip(bars, det_accs):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## Part 7: Training-time OOD Methods

These methods improve OOD detection **during training** by modifying how the model learns.

### 7.1 Mixup Training

Mixup creates virtual training examples by linear interpolation:
- $\tilde{x} = \lambda x_i + (1-\lambda) x_j$
- $\tilde{y} = \lambda y_i + (1-\lambda) y_j$

This smooths decision boundaries and improves OOD detection.

In [ ]:
seed_everything(42)
model_mixup = ConvNet(num_classes=10, dropout_rate=0.0).to(device)
optimizer = torch.optim.Adam(model_mixup.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

print("Training with Mixup (alpha=1.0)...")
model_mixup.train()
for epoch in range(10):
    total_loss = 0
    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        
        # Apply mixup
        mixed_data, y_a, y_b, lam = mixup_data(data, target, alpha=1.0)
        
        optimizer.zero_grad()
        output = model_mixup(mixed_data)
        
        # Mixup loss
        loss = mixup_criterion(criterion, output, y_a, y_b, lam)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if (epoch + 1) % 5 == 0:
        print(f"  Epoch {epoch+1}/10 - Loss: {total_loss/len(train_loader):.4f}")

print("Done!")

### 7.2 CutMix Training

CutMix cuts and pastes patches between images instead of linear blending.

In [ ]:
seed_everything(42)
model_cutmix = ConvNet(num_classes=10, dropout_rate=0.0).to(device)
optimizer = torch.optim.Adam(model_cutmix.parameters(), lr=0.001)
cutmix = CutMix(alpha=1.0)
criterion = nn.CrossEntropyLoss()

print("Training with CutMix (alpha=1.0)...")
model_cutmix.train()
for epoch in range(10):
    total_loss = 0
    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        
        # Apply CutMix
        mixed_data, y_a, y_b, lam = cutmix(data, target)
        
        optimizer.zero_grad()
        output = model_cutmix(mixed_data)
        loss = lam * criterion(output, y_a) + (1 - lam) * criterion(output, y_b)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if (epoch + 1) % 5 == 0:
        print(f"  Epoch {epoch+1}/10 - Loss: {total_loss/len(train_loader):.4f}")

print("Done!")

### 7.3 Outlier Exposure

Outlier Exposure trains the model to produce uniform predictions on auxiliary OOD data.
We use a small subset of MNIST digits as the auxiliary outliers.

In [ ]:
# Load auxiliary OOD data for training (different from test OOD)
aux_ood_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
# Use subset for efficiency
aux_ood_subset = Subset(aux_ood_dataset, range(0, 10000))
aux_ood_loader = DataLoader(aux_ood_subset, batch_size=256, shuffle=True, **loader_kwargs)

seed_everything(42)
model_oe = ConvNet(num_classes=10, dropout_rate=0.0).to(device)
optimizer = torch.optim.Adam(model_oe.parameters(), lr=0.001)
oe_criterion = OutlierExposureLoss(lambda_oe=0.5)

print("Training with Outlier Exposure (lambda=0.5)...")
model_oe.train()

for epoch in range(10):
    total_loss = 0
    ood_iter = iter(aux_ood_loader)
    
    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        
        # Get OOD batch (cycle if exhausted)
        try:
            ood_data, _ = next(ood_iter)
        except StopIteration:
            ood_iter = iter(aux_ood_loader)
            ood_data, _ = next(ood_iter)
        ood_data = ood_data.to(device)
        
        optimizer.zero_grad()
        logits_in = model_oe(data)
        logits_out = model_oe(ood_data)
        
        loss = oe_criterion(logits_in, target, logits_out)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if (epoch + 1) % 5 == 0:
        print(f"  Epoch {epoch+1}/10 - Loss: {total_loss/len(train_loader):.4f}")

print("Done!")

In [ ]:
# Compare training-time methods with baseline
training_models = {
    'Baseline': model,
    'Mixup': model_mixup,
    'CutMix': model_cutmix,
    'Outlier Exposure': model_oe,
}

print("Training-time Method Comparison (using Energy detector):")
print("=" * 55)
print(f"{'Method':<20} {'AUROC':>10} {'FPR@95TPR':>12}")
print("-" * 55)

for name, mdl in training_models.items():
    mdl.eval()
    detector = Energy(mdl, temperature=1.0)
    
    id_scores = score_loader(detector, test_id_loader, device)
    ood_scores = score_loader(detector, test_ood_loader, device)
    
    auc = auroc(id_scores, ood_scores)
    fpr = fpr_at_tpr(id_scores, ood_scores, tpr=0.95)
    
    print(f"{name:<20} {auc:>10.4f} {fpr:>12.4f}")

print("=" * 55)

## Part 8: Production Deployment

### 8.1 Threshold Selection

In [ ]:
# Use incerto's threshold utility
best_method = max(metrics_results, key=lambda k: metrics_results[k]['auroc'])
best_detector = detectors[best_method]
best_id_scores = results[best_method]['id_scores']

# Compute threshold at 95% ID acceptance rate
threshold = compute_threshold_at_tpr(best_id_scores, target_tpr=0.95)
print(f"Best method: {best_method}")
print(f"Threshold @95% ID acceptance: {threshold:.4f}")

# Get binary predictions using incerto's utility
id_preds = get_ood_predictions(best_id_scores, threshold)
ood_preds = get_ood_predictions(results[best_method]['ood_scores'], threshold)

print(f"\nID samples flagged as OOD: {id_preds.sum()} / {len(id_preds)} ({100*id_preds.mean():.1f}%)")
print(f"OOD samples correctly flagged: {ood_preds.sum()} / {len(ood_preds)} ({100*ood_preds.mean():.1f}%)")

### 8.2 Save and Load Detectors

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as tmpdir:
    # Save detector
    path = f"{tmpdir}/ood_detector.pt"
    best_detector.save(path)
    print(f"Saved {best_method} detector ({os.path.getsize(path)} bytes)")
    
    # Load detector (for methods that need fitting)
    if best_method in ['Mahalanobis', 'KNN']:
        loaded = type(best_detector).load(path, model, layer_name='fc1')
    else:
        loaded = type(best_detector)(model)
        loaded.load_state_dict(torch.load(path, weights_only=True))
    
    print(f"Loaded: {loaded}")

### 8.3 Production Inference Pipeline

In [ ]:
def predict_with_ood_detection(model, detector, inputs, threshold):
    """
    Production inference with OOD detection.
    
    Args:
        model: Classification model
        detector: OOD detector
        inputs: Input batch
        threshold: OOD score threshold
    
    Returns:
        dict with predictions, confidences, ood_scores, is_ood flags
    """
    model.eval()
    
    with torch.no_grad():
        # Classification
        logits = model(inputs)
        probs = F.softmax(logits, dim=-1)
        confidences, predictions = probs.max(dim=-1)
        
        # OOD detection
        ood_scores = detector.score(inputs)
        is_ood = ood_scores > threshold
    
    return {
        'predictions': predictions.cpu(),
        'confidences': confidences.cpu(),
        'ood_scores': ood_scores.cpu(),
        'is_ood': is_ood.cpu(),
    }

# Example: process a batch
fashion_labels = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
                  'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Boot']

print("Production Inference Examples:")
print("=" * 75)

# Test on ID samples
id_batch = torch.stack([test_id_dataset[i][0] for i in range(3)]).to(device)
id_labels = [test_id_dataset[i][1] for i in range(3)]
result = predict_with_ood_detection(model, best_detector, id_batch, threshold)

print("\nIn-Distribution samples (FashionMNIST):")
for i in range(3):
    pred = fashion_labels[result['predictions'][i]]
    true = fashion_labels[id_labels[i]]
    conf = result['confidences'][i].item()
    ood = result['is_ood'][i].item()
    status = "FLAGGED" if ood else "accepted"
    print(f"  Pred: {pred:10s} ({conf:.1%}) | True: {true:10s} | OOD: {status}")

# Test on OOD samples
ood_batch = torch.stack([test_ood_dataset[i][0] for i in range(3)]).to(device)
ood_labels = [test_ood_dataset[i][1] for i in range(3)]
result = predict_with_ood_detection(model, best_detector, ood_batch, threshold)

print("\nOut-of-Distribution samples (MNIST digits):")
for i in range(3):
    pred = fashion_labels[result['predictions'][i]]
    digit = ood_labels[i]
    conf = result['confidences'][i].item()
    ood = result['is_ood'][i].item()
    status = "FLAGGED" if ood else "missed"
    print(f"  Pred: {pred:10s} ({conf:.1%}) | True: digit {digit}  | OOD: {status}")

print("=" * 75)

## Part 9: Best Practices

### Method Selection Guidelines

| Situation | Recommended |
|-----------|-------------|
| Fastest inference | `MSP` or `MaxLogit` |
| Good speed/accuracy balance | `Energy` |
| Best accuracy (have compute) | `Mahalanobis` or `KNN` |
| Need input robustness | `ODIN` (with tuning) |
| Have auxiliary OOD data | `OutlierExposureLoss` |
| Want smoother boundaries | `mixup_data` or `CutMix` |

### Hyperparameter Tuning

- **ODIN**: Tune `temperature` (100-1000) and `epsilon` (0.001-0.01)
- **Energy**: Tune `temperature` (0.1-10)
- **Mahalanobis/KNN**: Try different feature layers
- **KNN**: Tune `k` based on training set size (typically 10-100)

### Production Checklist

- [ ] Validate on representative OOD data from your domain
- [ ] Set threshold using validation set
- [ ] Monitor OOD detection rate in production
- [ ] Log flagged samples for human review
- [ ] Consider ensemble of multiple detectors for critical applications

### References

1. **MSP**: Hendrycks & Gimpel, "A Baseline for Detecting Misclassified and OOD Examples" (ICLR 2017)
2. **ODIN**: Liang et al., "Enhancing The Reliability of OOD Image Detection" (ICLR 2018)
3. **Mahalanobis**: Lee et al., "A Simple Unified Framework for Detecting OOD Samples" (NeurIPS 2018)
4. **Energy**: Liu et al., "Energy-based Out-of-distribution Detection" (NeurIPS 2020)
5. **KNN**: Sun et al., "Out-of-Distribution Detection with Deep Nearest Neighbors" (NeurIPS 2022)
6. **Mixup**: Zhang et al., "mixup: Beyond Empirical Risk Minimization" (ICLR 2018)
7. **Outlier Exposure**: Hendrycks et al., "Deep Anomaly Detection with Outlier Exposure" (ICLR 2019)

## Summary

In this notebook, you learned:

1. **6 post-hoc OOD detection methods** - MSP, Energy, MaxLogit, ODIN, Mahalanobis, KNN
2. **4 training-time methods** - Mixup, CutMix, Outlier Exposure, Energy Regularization
3. **3 evaluation metrics** - AUROC, FPR@95TPR, Detection Accuracy
4. **Visualization tools** - `plot_roc`, `score_hist`
5. **Production utilities** - `compute_threshold_at_tpr`, `get_ood_predictions`, save/load
